# Train (Colab)

Fine-tunes VideoMAE-Base or Video Swin-Tiny on ASL Citizen.

**Colab sessions end without warning.** This notebook is built around that: the dataset is staged to ephemeral disk, checkpoints go to Drive, and re-running the training cell resumes rather than restarting. See D-004 and D-007 in `docs/DECISIONS.md`.

**Runtime → Change runtime type → GPU** before running.

Prerequisite: the Phase 2A audit has been run and the manifests and label map exist on Drive.

## 1. Code and persistent storage

In [ ]:
import os

from google.colab import drive

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

!git clone -q $REPO_URL /content/asl || (cd /content/asl && git pull -q)
PROJECT = "/content/asl/ASL_training"

# Checkpoints and run outputs must survive session termination.
drive.mount("/content/drive")
PERSISTENT = "/content/drive/MyDrive/asl-training"
ARTIFACTS = f"{PERSISTENT}/artifacts"
OUTPUTS = f"{PERSISTENT}/outputs"

os.makedirs(OUTPUTS, exist_ok=True)
print(f"project    {PROJECT}")
print(f"artifacts  {ARTIFACTS}")
print(f"outputs    {OUTPUTS}")

In [ ]:
!pip install -q -e $PROJECT --no-deps
!pip install -q av

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
print(torch.cuda.get_device_name(0))

## 2. Stage the dataset to local disk

Raw video stays on ephemeral `/content`. Decoding thousands of clips per epoch across mounted Drive is far too slow and will hit Drive rate limits.

This cost is paid once per session. Skipped automatically if the data is already staged.

In [ ]:
DATASET_ROOT = "/content/asl_citizen"
KAGGLE_DATASET = ""  # e.g. "someuser/asl-citizen"

if os.path.isdir(f"{DATASET_ROOT}/videos"):
    print("already staged")
else:
    assert KAGGLE_DATASET, "Set KAGGLE_DATASET to the mirror slug."

    # Upload kaggle.json via the Files pane, or place it on Drive.
    os.makedirs("/root/.config/kaggle", exist_ok=True)
    !cp "$PERSISTENT/kaggle.json" /root/.config/kaggle/kaggle.json
    !chmod 600 /root/.config/kaggle/kaggle.json
    !pip install -q kaggle

    os.makedirs(DATASET_ROOT, exist_ok=True)
    !kaggle datasets download -d $KAGGLE_DATASET -p /content --unzip -q

!df -h /content | tail -1
!ls $DATASET_ROOT | head

## 3. Verify what was staged

A partial extraction that silently yields fewer videos would violate the no-silent-reduction rule. Training validates manifests before starting, but checking here fails faster.

In [ ]:
import json

audit_path = f"{ARTIFACTS}/audits/asl_citizen_audit.json"
assert os.path.exists(audit_path), (
    f"No audit at {audit_path}. Run notebooks/kaggle/01_dataset_audit.ipynb first."
)

with open(audit_path) as handle:
    audit = json.load(handle)

print(f"audited records {audit['counts']['manifest_records']}")
print(f"classes         {audit['counts']['classes']}")
print(f"label map       {audit['label_map_identity']}")

staged = sum(len(files) for _, _, files in os.walk(f"{DATASET_ROOT}"))
print(f"\nstaged files    {staged}")

if audit["problems"]:
    print(f"\nThe audit reported {len(audit['problems'])} problem(s):")
    for problem in audit["problems"]:
        print(f"  - {problem}")

## 4. Train

Re-running this cell after a disconnect **resumes** from the last checkpoint. It does not restart, and it does not create a second run.

Pick the model config to switch architectures; everything else stays identical, which is what keeps the Phase 5 comparison fair.

In [ ]:
MODEL = "videomae_base"  # or "video_swin_tiny"
EXPERIMENT = "exp-001-videomae-baseline"
RUN_NAME = "videomae-baseline-seed42"

# VideoMAE-Base is ~3x the size of Swin-Tiny, so it needs a smaller physical
# batch. Raise accumulation to keep the effective batch comparable across
# architectures and across whatever GPU Colab assigns.
BATCH_SIZE = 8

!cd $PROJECT && python scripts/train.py \
    --model-config configs/models/$MODEL.yaml \
    --training-config configs/training/baseline.yaml \
    --artifacts-dir "$ARTIFACTS" \
    --dataset-root "$DATASET_ROOT" \
    --output-root "$OUTPUTS" \
    --experiment $EXPERIMENT \
    --run-name $RUN_NAME \
    --batch-size $BATCH_SIZE

## 5. Progress so far

Safe to run mid-training or after a disconnect.

In [ ]:
run_dir = f"{OUTPUTS}/{EXPERIMENT}/{RUN_NAME}"

with open(f"{run_dir}/history.json") as handle:
    history = json.load(handle)

for entry in history:
    line = f"epoch {entry['epoch']:3d}  loss {entry['train_loss']:.4f}"
    for name, value in entry["validation"].items():
        line += f"  {name} {value:.4f}"
    line += f"  ({entry['duration_seconds']:.0f}s)"
    if entry["non_finite_losses"]:
        line += f"  [{entry['non_finite_losses']} non-finite]"
    print(line)

with open(f"{run_dir}/run_metadata.json") as handle:
    metadata = json.load(handle)

print(f"\nrun kind        {metadata['run_kind']}")
print(f"effective batch {metadata['effective_batch_size']}")
print(
    f"precision       {metadata['precision_active']} (requested {metadata['precision_requested']})"
)
print(f"gpu             {metadata['environment'].get('gpu')}")

## Notes

**On disconnect:** reconnect, re-run cells 1 through 4. Staging repeats; training continues from the last checkpoint.

**On out-of-memory:** Colab may assign a T4, an L4, or an A100. Lower `--batch-size` and raise `gradient_accumulation_steps` in the training config by the same factor, so the effective batch stays comparable. Changing only the batch size makes the run non-comparable to earlier ones.

**Checkpoints on Drive:** `latest.pt` resumes, `best.pt` is for evaluation. The previous `latest` is retained, so an interrupted write cannot strand the run.

**The test split is not touched here.** Final test evaluation is a separate, deliberate step after model and threshold selection are fixed.